# 1 DATA PREPARATION

**Project:** Irish Milk Supply: Growth, Seasonality and Forecasting

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the raw data, checks its quality and turns it into one clean monthly table that the rest of the project uses for analysis, SQL and the Power BI dashboard.

### Data sources
| Dataset | Source | Content |
|---|---|---|
| AKM01 | CSO | Milk intake by milk processors and co-ops; fat and protein content |
| AKM02 | CSO | Milk sales for human consumption (drinking milk) |
| AKM03 | CSO | Production of butter, cheese and skimmed milk powder |
| 32-10-0113-01 | Statistics Canada | Canadian milk production, used for comparison |

### Contents
1. AKM01: inspection and cleaning
2. AKM02: inspection and cleaning
3. AKM03: inspection and cleaning
4. Statistics Canada: inspection and cleaning
5. Combining into one monthly table
6. Saving the clean data

### Output
This notebook saves the cleaned data to `data/processed/ireland_dairy_monthly.csv` (one row per month, one column per measure).

## Setup

In [1]:
import pandas as pd
import numpy as np

# Show all columns when displaying a DataFrame
pd.set_option("display.max_columns", None)

## 1. AKM01: Milk intake, fat and protein content

**Source:** Central Statistics Office (CSO), table AKM01 "Intake of Cows Milk by Milk Processors and Co-Ops" (earlier titled "Intake of Cows Milk by Creameries and Pasteurisers"). Downloaded from data.cso.ie in October 2026. Licence: CC BY 4.0.

**Structure according to the source:** monthly figures for three statistics (milk intake, fat content and protein content), each split by Domestic and Import source.

In this section I inspect the raw file before cleaning it.

### 1.1 Inspection

#### Load the file and check its size

In [2]:
# Load the raw file
akm01 = pd.read_csv("../data/raw/AKM01.csv")

# Number of rows and columns
akm01.shape

(3720, 5)

#### First and last rows

In [3]:
# First and last 3 rows: format of the data, and where it starts and ends
display(akm01.head(3))
display(akm01.tail(3))

,Statistic Label,Month,Domestic or Import Source,UNIT,VALUE
0,Intake of Cows Milk by Milk Processors and Co-Ops,1975 January,Domestic,Million Litres,55.8
1,Intake of Cows Milk by Milk Processors and Co-Ops,1975 January,Import,Million Litres,NaN
2,Intake of Cows Milk by Milk Processors and Co-Ops,1975 February,Domestic,Million Litres,84.4


,Statistic Label,Month,Domestic or Import Source,UNIT,VALUE
3717,Protein Content,2026 July,Import,%,NaN
3718,Protein Content,2026 August,Domestic,%,3.61
3719,Protein Content,2026 August,Import,%,NaN


First look: the data is in *long* format, with one value per row. The first rows are milk intake for January and February 1975, with one Domestic and one Import row per month. The last rows are protein content for 2026.

#### Column types and missing counts

In [4]:
# Column names, data types and non-missing counts
akm01.info()

<class 'pandas.DataFrame'>
RangeIndex: 3720 entries, 0 to 3719
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Statistic Label            3720 non-null   str    
 1   Month                      3720 non-null   str    
 2   Domestic or Import Source  3720 non-null   str    
 3   UNIT                       3720 non-null   str    
 4   VALUE                      1991 non-null   float64
dtypes: float64(1), str(4)
memory usage: 145.4 KB


The first four columns are text (`str`) and complete. `VALUE` is already stored as numbers (`float64`), but it has only 1,991 non-missing values out of 3,720.

#### Categories in each column

In [5]:
# Number of distinct values in each column
akm01.nunique()

Statistic Label                3
Month                        620
Domestic or Import Source      2
UNIT                           2
VALUE                        972
dtype: int64

In [6]:
# Categories in each descriptive column
print(akm01["Statistic Label"].value_counts(), "\n")
print(akm01["Domestic or Import Source"].value_counts(), "\n")
print(akm01["UNIT"].value_counts())

Statistic Label
Intake of Cows Milk by Milk Processors and Co-Ops    1240
Fat Content                                          1240
Protein Content                                      1240
Name: count, dtype: int64 

Domestic or Import Source
Domestic    1860
Import      1860
Name: count, dtype: int64 

UNIT
%                 2480
Million Litres    1240
Name: count, dtype: int64


There are 3 statistics (1,240 rows each) and 2 sources (1,860 rows each). Milk intake is measured in million litres; fat and protein in %. `Month` has 620 distinct values (January 1975 to August 2026).

#### Check that every month is complete

In [7]:
# Number of rows for each month (first 3 shown)
akm01["Month"].value_counts().head(3)

Month
1975 January     6
1975 February    6
1975 March       6
Name: count, dtype: int64

In [8]:
# Check that every month has the same number of rows
akm01["Month"].value_counts().unique()

array([6])

In [9]:
# Check that no statistic, month and source combination appears twice
akm01.duplicated(subset=["Statistic Label", "Month", "Domestic or Import Source"]).sum()

np.int64(0)

Structure: all 620 months appear exactly 6 times (3 statistics × 2 sources). The file is a complete grid with no missing or duplicated rows: 620 × 6 = 3,720 rows.

#### Check the `VALUE` column

`.info()` shows that `VALUE` is already numeric. Older versions of this table used "." for values that were not available, so I still check for text entries before counting the missing values.

In [10]:
# Count empty cells in VALUE in the original file
akm01["VALUE"].isna().sum()

np.int64(1729)

In [11]:
# Convert VALUE to numbers in a separate variable (akm01 is not changed)
# Anything that can't be converted becomes NaN
vals = pd.to_numeric(akm01["VALUE"], errors="coerce")

# Count missing values after the conversion
vals.isna().sum()

np.int64(1729)

The number of missing values is the same before and after the conversion (1,729), so there are no text entries hidden in `VALUE`.

In [12]:
# Show the values that were filled in but are not numbers
akm01.loc[vals.isna() & akm01["VALUE"].notna(), "VALUE"].value_counts()

Series([], Name: count, dtype: int64)

In [13]:
# Count usable numbers
vals.notna().sum()

np.int64(1991)

Summary of `VALUE`:
- 1,991 cells contain numbers
- 1,729 cells are empty
- Total: 3,720

There are no text entries, so the only issue is the empty cells.

#### Where the missing values are

I count the missing values for each statistic and source.

In [14]:
# Count missing values for each statistic and source
akm01[vals.isna()].groupby(["Statistic Label", "Domestic or Import Source"]).size()

Statistic Label                                    Domestic or Import Source
Fat Content                                        Domestic                      60
                                                   Import                       620
Intake of Cows Milk by Milk Processors and Co-Ops  Import                       369
Protein Content                                    Domestic                      60
                                                   Import                       620
dtype: int64

Imported milk has no fat or protein data at all (620 missing each, which is every month). Imported milk intake has 369 missing months. Domestic fat and protein have 60 missing months each. Domestic milk intake does not appear in the list, so it has no missing values.

In [15]:
# Convert the Month text to real dates
dates = pd.to_datetime(akm01["Month"], format="%Y %B")
dates.head(3)

0   1975-01-01
1   1975-01-01
2   1975-02-01
Name: Month, dtype: datetime64[us]

In [16]:
# Keep only rows that have a number, and add the dates as a temporary column
with_data = akm01[vals.notna()].assign(date=dates)

# First and last month with data for each statistic and source
with_data.groupby(["Statistic Label", "Domestic or Import Source"])["date"].agg(["min", "max"])

min  \
Statistic Label                                   Domestic or Import Source              
Fat Content                                       Domestic                  1980-01-01   
Intake of Cows Milk by Milk Processors and Co-Ops Domestic                  1975-01-01   
                                                  Import                    1998-01-01   
Protein Content                                   Domestic                  1980-01-01   

                                                                                   max  
Statistic Label                                   Domestic or Import Source             
Fat Content                                       Domestic                  2026-08-01  
Intake of Cows Milk by Milk Processors and Co-Ops Domestic                  2026-08-01  
                                                  Import                    2019-09-01  
Protein Content                                   Domestic                  2026-08-01

Domestic milk intake covers the full period, January 1975 to August 2026. Domestic fat and protein start in January 1980, which explains their 60 missing months (1975–1979). Imported milk intake is only reported from January 1998 to September 2019. The 369 missing months are mostly outside that period (359 months), plus 10 gaps inside it. Imported fat and protein do not appear because they have no data at all.

#### Summary of findings

AKM01 has a complete structure: 620 months from 1975 to 2026, 6 rows per month, no duplicates. `VALUE` is numeric and has no text entries. The missing values are not errors. They come from data CSO doesn't publish, mainly for imported milk. Domestic milk intake has no gaps.

For cleaning, I will convert `Month` to dates, keep only domestic data, and reshape the table to one row per month.

### 1.2 Cleaning

#### Make sure `VALUE` is numeric

In [17]:
# Work on a copy so the raw data stays unchanged
akm01_clean = akm01.copy()

# Make sure VALUE is numeric. It already is in the latest file, but this step
# also turns any text markers (such as ".") into NaN if CSO uses them again
akm01_clean["VALUE"] = pd.to_numeric(akm01_clean["VALUE"], errors="coerce")

akm01_clean["VALUE"].dtype

dtype('float64')

#### Convert `Month` to dates

In [18]:
# Convert the Month text to real dates in a new column
akm01_clean["date"] = pd.to_datetime(akm01_clean["Month"], format="%Y %B")

# Compare the original text and the new date
akm01_clean[["Month", "date"]].head(3)

,Month,date
0,1975 January,1975-01-01
1,1975 January,1975-01-01
2,1975 February,1975-02-01


#### Keep only domestic data
The import figures have no fat or protein data, and intake is only reported for 1998–2019, so I keep domestic data only.

In [19]:
# Keep only domestic rows
akm01_clean = akm01_clean[akm01_clean["Domestic or Import Source"] == "Domestic"]

# Check the result
print(akm01_clean.shape)
print(akm01_clean["Domestic or Import Source"].unique())

(1860, 6)
<StringArray>
['Domestic']
Length: 1, dtype: str


#### Reshape to one row per month
At the moment each month has 3 rows (intake, fat and protein). I turn the 3 statistics into columns, so each month becomes a single row.

In [20]:
# Turn the statistics into columns: one row per month
akm01_wide = akm01_clean.pivot(index="date", columns="Statistic Label", values="VALUE")

print(akm01_wide.shape)
akm01_wide.head(3)

(620, 3)


Statistic Label,Fat Content,Intake of Cows Milk by Milk Processors and Co-Ops,Protein Content
date,,,
1975-01-01,NaN,55.8,NaN
1975-02-01,NaN,84.4,NaN
1975-03-01,NaN,193.3,NaN


#### Rename the columns
The CSO names are long, so I rename them to short names that include the unit.

In [21]:
# Short column names with units
akm01_wide = akm01_wide.rename(columns={
    "Intake of Cows Milk by Milk Processors and Co-Ops": "intake_ml",
    "Fat Content": "fat_pct",
    "Protein Content": "protein_pct",
})

# Remove the leftover "Statistic Label" name above the columns
akm01_wide.columns.name = None

akm01_wide.head(3)

,fat_pct,intake_ml,protein_pct
date,,,
1975-01-01,NaN,55.8,NaN
1975-02-01,NaN,84.4,NaN
1975-03-01,NaN,193.3,NaN


#### Check the cleaned table

In [22]:
# Put the columns in a logical order
akm01_wide = akm01_wide[["intake_ml", "fat_pct", "protein_pct"]]

# Missing values in each column
akm01_wide.isna().sum()

intake_ml       0
fat_pct        60
protein_pct    60
dtype: int64

In [23]:
# Summary statistics to check the values make sense
akm01_wide.describe()

,intake_ml,fat_pct,protein_pct
count,620.000000,560.000000,560.000000
mean,470.324161,3.879696,3.353518
std,260.374638,0.371308,0.239157
min,55.800000,3.280000,2.860000
25%,220.400000,3.587500,3.190000
50%,467.000000,3.820000,3.310000
75%,652.175000,4.100000,3.500000
max,1183.400000,5.120000,4.040000


All values are in a realistic range. Monthly intake varies from 56 to 1,183 million litres, which reflects the strong seasonal pattern of Irish milk production. Fat is between 3.3% and 5.1% and protein between 2.9% and 4.0%.

#### Cleaning summary

I made sure `VALUE` is numeric, converted `Month` to dates, kept only domestic data, and reshaped the table to one row per month. The result, `akm01_wide`, has 620 rows (January 1975 to August 2026) and 3 columns: `intake_ml`, `fat_pct` and `protein_pct`. Fat and protein are missing for 1975–1979 because CSO starts reporting them in 1980.

## 2. AKM02: Milk sales for human consumption

Source: Central Statistics Office (CSO), table AKM02 "Milk Sales (Dairy) for Human Consumption". Downloaded from data.cso.ie in October 2026. Licence: CC BY 4.0.

In this section I inspect the raw file before cleaning it.

### 2.1 Inspection

#### Load the file and check its size

In [24]:
# Load the raw file
akm02 = pd.read_csv("../data/raw/AKM02.csv")

# Number of rows and columns
akm02.shape

(2800, 5)

#### First and last rows

In [25]:
# First and last 3 rows: format of the data, and where it starts and ends
display(akm02.head(3))
display(akm02.tail(3))

,STATISTIC Label,Month,Type of Milk,UNIT,VALUE
0,Milk Sales (Dairy) for Human Consumption,1980 January,All Milk,Million Litres,38.2
1,Milk Sales (Dairy) for Human Consumption,1980 January,Whole Milk,Million Litres,38.2
2,Milk Sales (Dairy) for Human Consumption,1980 January,Skimmed and Semi-skimmed milk,Million Litres,NaN


,STATISTIC Label,Month,Type of Milk,UNIT,VALUE
2797,Milk Sales (Dairy) for Human Consumption,2026 August,Skimmed and Semi-skimmed milk,Million Litres,13.0
2798,Milk Sales (Dairy) for Human Consumption,2026 August,Skimmed Milk,Million Litres,NaN
2799,Milk Sales (Dairy) for Human Consumption,2026 August,Semi-skimmed milk,Million Litres,NaN


First look: the data is in long format. The first column is called `STATISTIC Label` (capital letters, unlike AKM01). Instead of a source column there is `Type of Milk`. The data runs from January 1980 to August 2026, and the rows for each month are grouped together.

#### Column types and missing counts

In [26]:
# Column names, data types and non-missing counts
akm02.info()

<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   STATISTIC Label  2800 non-null   str    
 1   Month            2800 non-null   str    
 2   Type of Milk     2800 non-null   str    
 3   UNIT             2800 non-null   str    
 4   VALUE            2394 non-null   float64
dtypes: float64(1), str(4)
memory usage: 109.5 KB


The first four columns are text and complete. `VALUE` is numeric (`float64`) and has 2,394 non-missing values out of 2,800.

#### Categories in each column

In [27]:
# Number of distinct values in each column
akm02.nunique()

STATISTIC Label      1
Month              560
Type of Milk         5
UNIT                 1
VALUE              439
dtype: int64

In [28]:
# Categories in each descriptive column
print(akm02["STATISTIC Label"].value_counts(), "\n")
print(akm02["Type of Milk"].value_counts(), "\n")
print(akm02["UNIT"].value_counts())

STATISTIC Label
Milk Sales (Dairy) for Human Consumption    2800
Name: count, dtype: int64 

Type of Milk
All Milk                         560
Whole Milk                       560
Skimmed and Semi-skimmed milk    560
Skimmed Milk                     560
Semi-skimmed milk                560
Name: count, dtype: int64 

UNIT
Million Litres    2800
Name: count, dtype: int64


There is only one statistic (milk sales for human consumption) and one unit (million litres). `Type of Milk` has 5 categories with 560 rows each. "All Milk" is the total for all types. `Month` has 560 distinct values (January 1980 to August 2026).

#### Check that every month is complete

In [29]:
# Number of rows for each month (first 3 shown)
akm02["Month"].value_counts().head(3)

Month
1980 January     5
1980 February    5
1980 March       5
Name: count, dtype: int64

In [30]:
# Check that every month has the same number of rows
akm02["Month"].value_counts().unique()

array([5])

In [31]:
# Check that no month and milk type combination appears twice
akm02.duplicated(subset=["Month", "Type of Milk"]).sum()

np.int64(0)

Structure: all 560 months appear exactly 5 times (one row per milk type), with no duplicates. The file is a complete grid: 560 × 5 = 2,800 rows.

#### Check the `VALUE` column

`VALUE` is already numeric. I run the same checks as in AKM01 to confirm there are no text entries.

In [32]:
# Count empty cells in VALUE in the original file
akm02["VALUE"].isna().sum()

np.int64(406)

In [33]:
# Convert VALUE to numbers in a separate variable (akm02 is not changed)
# Anything that can't be converted becomes NaN
vals2 = pd.to_numeric(akm02["VALUE"], errors="coerce")

# Count missing values after the conversion
vals2.isna().sum()

np.int64(406)

In [34]:
# Show the values that were filled in but are not numbers
akm02.loc[vals2.isna() & akm02["VALUE"].notna(), "VALUE"].value_counts()

Series([], Name: count, dtype: int64)

In [35]:
# Count usable numbers
vals2.notna().sum()

np.int64(2394)

Summary of `VALUE`:
- 2,394 cells contain numbers
- 406 cells are empty
- Total: 2,800

As in AKM01, there are no text entries.

#### Where the missing values are

In [36]:
# Count missing values for each milk type
akm02[vals2.isna()].groupby("Type of Milk").size()

Type of Milk
Semi-skimmed milk                131
Skimmed Milk                     131
Skimmed and Semi-skimmed milk    144
dtype: int64

In [37]:
# Convert the Month text to real dates
dates2 = pd.to_datetime(akm02["Month"], format="%Y %B")

# First and last month with data for each milk type
with_data2 = akm02[vals2.notna()].assign(date=dates2)
with_data2.groupby("Type of Milk")["date"].agg(["min", "max"])

,min,max
Type of Milk,,
All Milk,1980-01-01,2026-08-01
Semi-skimmed milk,1985-01-01,2020-09-01
Skimmed Milk,1985-01-01,2020-09-01
Skimmed and Semi-skimmed milk,1992-01-01,2026-08-01
Whole Milk,1980-01-01,2026-08-01


All Milk and Whole Milk are complete for January 1980 to August 2026. The other types were only reported for part of the period: Skimmed and Semi-skimmed separately from 1985 to September 2020, and combined from 1992. The missing values are the months outside these periods, so again they come from what CSO publishes, not from errors.

#### Summary of findings

AKM02 has a complete structure: 560 months from 1980 to 2026, 5 rows per month (one per milk type), no duplicates. `VALUE` is numeric with no text entries. All Milk is complete for the whole period. The other milk types are only reported for part of the period.

For cleaning, I will keep only All Milk, which is the total and has no gaps, and convert `Month` to dates.

### 2.2 Cleaning

#### Keep only All Milk

In [38]:
# Work on a copy and keep only the All Milk rows
akm02_clean = akm02[akm02["Type of Milk"] == "All Milk"].copy()

akm02_clean.shape

(560, 5)

#### Make sure `VALUE` is numeric and convert `Month` to dates

In [39]:
# Convert VALUE to numbers and Month to dates
akm02_clean["VALUE"] = pd.to_numeric(akm02_clean["VALUE"], errors="coerce")
akm02_clean["date"] = pd.to_datetime(akm02_clean["Month"], format="%Y %B")

akm02_clean[["Month", "date", "VALUE"]].head(3)

,Month,date,VALUE
0,1980 January,1980-01-01,38.2
5,1980 February,1980-02-01,36.6
10,1980 March,1980-03-01,38.4


#### Keep only the columns needed

In [40]:
# One row per month, with the date as the row label and a clear column name
akm02_wide = akm02_clean.set_index("date")[["VALUE"]].rename(columns={"VALUE": "liquid_sales_ml"})

akm02_wide.head(3)

,liquid_sales_ml
date,
1980-01-01,38.2
1980-02-01,36.6
1980-03-01,38.4


#### Check the cleaned table

In [41]:
# Missing values and summary statistics
print(akm02_wide.isna().sum(), "\n")
akm02_wide.describe()

liquid_sales_ml    0
dtype: int64 



,liquid_sales_ml
count,560.000000
mean,42.645411
std,2.410526
min,36.200000
25%,41.000000
50%,42.900000
75%,44.300000
max,49.100000


No missing values. Monthly sales of drinking milk are between 36 and 49 million litres, so they are quite stable compared with milk intake.

#### Cleaning summary

I kept only All Milk, made sure `VALUE` is numeric, converted `Month` to dates, and kept one column. The result, `akm02_wide`, has 560 rows (January 1980 to August 2026) and one column, `liquid_sales_ml`, with no missing values.

## 3. AKM03: Production of dairy products

Source: Central Statistics Office (CSO), table AKM03 "Production of Dairy Products". Downloaded from data.cso.ie in October 2026. Licence: CC BY 4.0.

In this section I inspect the raw file before cleaning it.

### 3.1 Inspection

#### Load the file and check its size

In [42]:
# Load the raw file
akm03 = pd.read_csv("../data/raw/AKM03.csv")

# Number of rows and columns
akm03.shape

(1680, 5)

#### First and last rows

In [43]:
# First and last 3 rows: format of the data, and where it starts and ends
display(akm03.head(3))
display(akm03.tail(3))

,Statistic Label,Month,Product,UNIT,VALUE
0,Production of Dairy Products,1980 January,Cheese,000 Tonnes,0.0
1,Production of Dairy Products,1980 January,Butter,000 Tonnes,1.8
2,Production of Dairy Products,1980 January,Skimmed Milk Powder,000 Tonnes,1.6


,Statistic Label,Month,Product,UNIT,VALUE
1677,Production of Dairy Products,2026 August,Cheese,000 Tonnes,NaN
1678,Production of Dairy Products,2026 August,Butter,000 Tonnes,29.8
1679,Production of Dairy Products,2026 August,Skimmed Milk Powder,000 Tonnes,13.6


First look: the data is in long format, with a `Product` column (cheese, butter and skimmed milk powder). The unit is "000 Tonnes", which means thousands of tonnes. The data runs from January 1980 to August 2026, and the rows for each month are grouped together. In August 2026, cheese is already missing.

#### Column types and missing counts

In [44]:
# Column names, data types and non-missing counts
akm03.info()

<class 'pandas.DataFrame'>
RangeIndex: 1680 entries, 0 to 1679
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Statistic Label  1680 non-null   str    
 1   Month            1680 non-null   str    
 2   Product          1680 non-null   str    
 3   UNIT             1680 non-null   str    
 4   VALUE            1632 non-null   float64
dtypes: float64(1), str(4)
memory usage: 65.8 KB


The first four columns are text and complete. `VALUE` is numeric (`float64`) and has 1,632 non-missing values out of 1,680.

#### Categories in each column

In [45]:
# Number of distinct values in each column
akm03.nunique()

Statistic Label      1
Month              560
Product              3
UNIT                 1
VALUE              355
dtype: int64

In [46]:
# Categories in each descriptive column
print(akm03["Statistic Label"].value_counts(), "\n")
print(akm03["Product"].value_counts(), "\n")
print(akm03["UNIT"].value_counts())

Statistic Label
Production of Dairy Products    1680
Name: count, dtype: int64 

Product
Cheese                 560
Butter                 560
Skimmed Milk Powder    560
Name: count, dtype: int64 

UNIT
000 Tonnes    1680
Name: count, dtype: int64


There is only one statistic (production of dairy products) and one unit (thousand tonnes). In this table the first column is called `Statistic Label`, like AKM01 and unlike AKM02. `Product` has 3 categories with 560 rows each: cheese, butter and skimmed milk powder. `Month` has 560 distinct values (January 1980 to August 2026).

#### Check that every month is complete

In [47]:
# Number of rows for each month (first 3 shown)
akm03["Month"].value_counts().head(3)

Month
1980 January     3
1980 February    3
1980 March       3
Name: count, dtype: int64

In [48]:
# Check that every month has the same number of rows
akm03["Month"].value_counts().unique()

array([3])

In [49]:
# Check that no month and product combination appears twice
akm03.duplicated(subset=["Month", "Product"]).sum()

np.int64(0)

Structure: all 560 months appear exactly 3 times (one row per product), with no duplicates. The file is a complete grid: 560 × 3 = 1,680 rows.

#### Check the `VALUE` column

`VALUE` is already numeric. I run the same checks as before to confirm there are no text entries.

In [50]:
# Count empty cells in VALUE in the original file
akm03["VALUE"].isna().sum()

np.int64(48)

In [51]:
# Convert VALUE to numbers in a separate variable (akm03 is not changed)
# Anything that can't be converted becomes NaN
vals3 = pd.to_numeric(akm03["VALUE"], errors="coerce")

# Count missing values after the conversion
vals3.isna().sum()

np.int64(48)

In [52]:
# Show the values that were filled in but are not numbers
akm03.loc[vals3.isna() & akm03["VALUE"].notna(), "VALUE"].value_counts()

Series([], Name: count, dtype: int64)

In [53]:
# Count usable numbers
vals3.notna().sum()

np.int64(1632)

Summary of `VALUE`:
- 1,632 cells contain numbers
- 48 cells are empty
- Total: 1,680

As in the other tables, there are no text entries.

#### Where the missing values are

In [54]:
# Count missing values for each product
akm03[vals3.isna()].groupby("Product").size()

Product
Cheese                  8
Skimmed Milk Powder    40
dtype: int64

In [55]:
# First and last month with data for each product
dates3 = pd.to_datetime(akm03["Month"], format="%Y %B")
with_data3 = akm03[vals3.notna()].assign(date=dates3)
with_data3.groupby("Product")["date"].agg(["min", "max"])

,min,max
Product,,
Butter,1980-01-01,2026-08-01
Cheese,1980-01-01,2025-12-01
Skimmed Milk Powder,1980-01-01,2026-08-01


Butter is complete. Cheese stops in December 2025, so its 8 missing values are January to August 2026. Skimmed milk powder (SMP) has 40 missing values spread across the whole period, so I check in which months they occur.

In [56]:
# In which calendar months are the SMP values missing
smp_missing = akm03[vals3.isna() & (akm03["Product"] == "Skimmed Milk Powder")]
pd.to_datetime(smp_missing["Month"], format="%Y %B").dt.month.value_counts().sort_index()

Month
1      9
2      6
10     4
11    13
12     8
Name: count, dtype: int64

In [57]:
# Typical SMP production in each calendar month (median, thousand tonnes)
smp = with_data3[with_data3["Product"] == "Skimmed Milk Powder"]
pd.to_numeric(smp["VALUE"]).groupby(smp["date"].dt.month).median()

date
1      3.15
2      3.90
3      7.10
4     13.20
5     20.60
6     18.40
7     14.70
8     12.70
9      8.20
10     5.10
11     2.50
12     3.15
Name: VALUE, dtype: float64

All the missing SMP values are in October to February, which are the months when very little SMP is produced (median around 2.5–5 thousand tonnes, against about 21 in May). So the gaps are not random. If I filled them with an average, I would add production that probably did not happen. I leave them as missing.

#### Summary of findings

AKM03 has a complete structure: 560 months from 1980 to 2026, 3 rows per month (one per product), no duplicates. `VALUE` is numeric with no text entries. Butter is complete. Cheese has no data for 2026 yet. Skimmed milk powder has 40 missing months, all in October to February, when production is low.

For cleaning, I will convert `Month` to dates and reshape the table to one row per month, with one column per product. I will not fill the missing values.

### 3.2 Cleaning

#### Make sure `VALUE` is numeric and convert `Month` to dates

In [58]:
# Work on a copy so the raw data stays unchanged
akm03_clean = akm03.copy()

# Convert VALUE to numbers and Month to dates
akm03_clean["VALUE"] = pd.to_numeric(akm03_clean["VALUE"], errors="coerce")
akm03_clean["date"] = pd.to_datetime(akm03_clean["Month"], format="%Y %B")

akm03_clean[["Month", "date", "Product", "VALUE"]].head(3)

,Month,date,Product,VALUE
0,1980 January,1980-01-01,Cheese,0.0
1,1980 January,1980-01-01,Butter,1.8
2,1980 January,1980-01-01,Skimmed Milk Powder,1.6


#### Reshape to one row per month

In [59]:
# Turn the products into columns: one row per month
akm03_wide = akm03_clean.pivot(index="date", columns="Product", values="VALUE")

akm03_wide.shape

(560, 3)

#### Rename the columns

In [60]:
# Short column names with units (kt = thousand tonnes)
akm03_wide = akm03_wide.rename(columns={
    "Butter": "butter_kt",
    "Cheese": "cheese_kt",
    "Skimmed Milk Powder": "smp_kt",
})
akm03_wide.columns.name = None

# Put the columns in a fixed order
akm03_wide = akm03_wide[["butter_kt", "cheese_kt", "smp_kt"]]

akm03_wide.head(3)

,butter_kt,cheese_kt,smp_kt
date,,,
1980-01-01,1.8,0.0,1.6
1980-02-01,4.4,0.2,4.4
1980-03-01,8.7,1.7,9.0


#### Check the cleaned table

In [61]:
# Missing values and summary statistics
print(akm03_wide.isna().sum(), "\n")
akm03_wide.describe()

butter_kt     0
cheese_kt     8
smp_kt       40
dtype: int64 



,butter_kt,cheese_kt,smp_kt
count,560.000000,552.000000,520.000000
mean,13.931393,11.972989,10.273308
std,7.806699,9.563701,7.412432
min,1.600000,0.000000,0.800000
25%,7.275000,3.475000,4.300000
50%,13.700000,10.100000,8.650000
75%,18.400000,18.325000,14.225000
max,36.400000,35.800000,34.700000


The missing values match the inspection. All values are in a realistic range (0 to about 36 thousand tonnes a month). Cheese has 9 months with 0 production, all in December or January between 1980 and 1989, when cheese making stopped over the winter. I keep these as 0 because CSO reports them as values, not as missing.

#### Cleaning summary

I made sure `VALUE` is numeric, converted `Month` to dates and reshaped the table to one row per month. The result, `akm03_wide`, has 560 rows (January 1980 to August 2026) and 3 columns: `butter_kt`, `cheese_kt` and `smp_kt`. The missing values are left as they are.

## 4. Statistics Canada: Milk production and utilization

Source: Statistics Canada, table 32-10-0113-01 "Milk production and utilization". Downloaded in October 2026.

I use this table only to compare the seasonal pattern of milk supply. Canada makes a useful contrast to Ireland: its dairy farming is mostly housed with year-round production, and supply is controlled by quotas, while Irish production is grass-based and follows the grazing season.

### 4.1 Inspection

In this section I inspect the raw file before cleaning it.

#### Load the file and check its size

In [62]:
# Load the raw file
canada_all = pd.read_csv("../data/raw/statcan_3210011301.csv")

# Number of rows and columns
canada_all.shape

(26715, 15)

#### First and last rows

In [63]:
# First and last 3 rows: format of the data, and where it starts and ends
display(canada_all.head(3))
display(canada_all.tail(3))

,REF_DATE,GEO,DGUID,Dairy distribution,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1946-01,Canada,2016A000011124,"Milk production, total",Kilolitres,195,units,0,v382152,1.1,363058.0,NaN,NaN,t,0
1,1946-01,Canada,2016A000011124,Fluid purposes,Kilolitres,195,units,0,v382156,1.5,159466.0,NaN,NaN,NaN,0
2,1946-02,Canada,2016A000011124,"Milk production, total",Kilolitres,195,units,0,v382152,1.1,347330.0,NaN,NaN,t,0


,REF_DATE,GEO,DGUID,Dairy distribution,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
26712,2026-06,British Columbia,2016A000259,"Milk sold off farms, total",Kilolitres,195,units,0,v382162,11.4,71048.0,NaN,NaN,NaN,0
26713,2026-06,British Columbia,2016A000259,Fluid purposes,Kilolitres,195,units,0,v382163,11.5,32563.0,NaN,NaN,NaN,0
26714,2026-06,British Columbia,2016A000259,Industrial purposes,Kilolitres,195,units,0,v382164,11.6,38484.0,NaN,NaN,NaN,0


First look: the format is different from the CSO tables. Dates are written as "1946-01" in `REF_DATE`, the categories are in `Dairy distribution`, and `VALUE` is in kilolitres (thousands of litres). There are many extra columns such as `DGUID`, `VECTOR` and `COORDINATE`. The first rows are for Canada from January 1946, but the last rows are for British Columbia, so the file also contains the provinces.

#### Regions in the file

In [64]:
# Number of rows for each region
canada_all["GEO"].value_counts()

GEO
Canada                       3093
Prince Edward Island         2373
Nova Scotia                  2373
New Brunswick                2373
Quebec                       2373
Ontario                      2373
Manitoba                     2373
Saskatchewan                 2373
Alberta                      2373
British Columbia             2373
Newfoundland and Labrador    2265
Name: count, dtype: int64

The file has 11 regions: Canada as a whole and 10 provinces. I only need the national figures, so I keep the rows where `GEO` is "Canada" (3,093 of 26,715 rows) and inspect those.

In [65]:
# Keep only the national totals for Canada
canada = canada_all[canada_all["GEO"] == "Canada"].copy()

canada.shape

(3093, 15)

#### Column types and missing counts

In [66]:
# Column names, data types and non-missing counts
canada.info()

<class 'pandas.DataFrame'>
Index: 3093 entries, 0 to 26684
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   REF_DATE            3093 non-null   str    
 1   GEO                 3093 non-null   str    
 2   DGUID               3093 non-null   str    
 3   Dairy distribution  3093 non-null   str    
 4   UOM                 3093 non-null   str    
 5   UOM_ID              3093 non-null   int64  
 6   SCALAR_FACTOR       3093 non-null   str    
 7   SCALAR_ID           3093 non-null   int64  
 8   VECTOR              3093 non-null   str    
 9   COORDINATE          3093 non-null   float64
 10  VALUE               3036 non-null   float64
 11  STATUS              57 non-null     str    
 12  SYMBOL              0 non-null      float64
 13  TERMINATED          915 non-null    str    
 14  DECIMALS            3093 non-null   int64  
dtypes: float64(3), int64(3), str(9)
memory usage: 386.6 KB


`VALUE` is already numeric, with 57 missing values. `STATUS` has 57 entries ("..", Statistics Canada's symbol for "not available"), which match the missing values. `SYMBOL` is completely empty, and `TERMINATED` marks series that are no longer updated.

#### Categories in each column

In [67]:
# Number of distinct values in each column
canada.nunique()

REF_DATE               966
GEO                      1
DGUID                    1
Dairy distribution       7
UOM                      1
UOM_ID                   1
SCALAR_FACTOR            2
SCALAR_ID                1
VECTOR                   7
COORDINATE               7
VALUE                 2710
STATUS                   1
SYMBOL                   0
TERMINATED               1
DECIMALS                 1
dtype: int64

In [68]:
# Categories in the main descriptive columns
print(canada["Dairy distribution"].value_counts(), "\n")
print(canada["UOM"].value_counts())

Dairy distribution
Fluid purposes                966
Milk sold off farms, total    606
Industrial purposes           606
Delivered as cream            447
Milk production, total        396
Farm home consumed             36
Fed to livestock               36
Name: count, dtype: int64 

UOM
Kilolitres    3093
Name: count, dtype: int64


Most columns have only one value (for example `GEO` is now always "Canada" and `UOM` is always "Kilolitres"), so they carry no information. `Dairy distribution` has 7 categories, with very different numbers of rows, so they cover different periods.

#### Date range for each category

In [69]:
# First and last month and number of rows for each category
canada.groupby("Dairy distribution")["REF_DATE"].agg(["min", "max", "count"])

,min,max,count
Dairy distribution,,,
Delivered as cream,1976-01,2013-03,447
Farm home consumed,1976-01,1978-12,36
Fed to livestock,1976-01,1978-12,36
Fluid purposes,1946-01,2026-06,966
Industrial purposes,1976-01,2026-06,606
"Milk production, total",1946-01,1978-12,396
"Milk sold off farms, total",1976-01,2026-06,606


In [70]:
# Missing values for each category
canada[canada["VALUE"].isna()].groupby("Dairy distribution").size()

Dairy distribution
Delivered as cream    57
dtype: int64

In [71]:
# Check that no month and category combination appears twice
canada.duplicated(subset=["REF_DATE", "Dairy distribution"]).sum()

np.int64(0)

Total milk sold off farms covers January 1976 to June 2026 (606 months), with no missing values and no duplicates. The 57 missing values are all in "Delivered as cream", which stopped in 2013. Some categories (such as "Milk production, total") stopped in 1978. For the comparison with Ireland I only need the total sold off farms.

#### Summary of findings

The Canadian file has a different format from the CSO tables and also contains the provinces, so I filtered it to Canada. `VALUE` is numeric. Total milk sold off farms covers January 1976 to June 2026 with no missing values or duplicates. Most of the 15 columns have only one value and carry no information.

For cleaning, I will keep only the total, convert `REF_DATE` to dates and convert kilolitres to million litres, so the units match the Irish data.

### 4.2 Cleaning

#### Keep only the total and the needed columns

In [72]:
# Keep only total milk sold off farms, with the date and the value
canada_clean = canada.loc[canada["Dairy distribution"] == "Milk sold off farms, total", ["REF_DATE", "VALUE"]].copy()

canada_clean.shape

(606, 2)

#### Convert dates and units

In [73]:
# Convert REF_DATE ("1976-01") to dates
canada_clean["date"] = pd.to_datetime(canada_clean["REF_DATE"], format="%Y-%m")

# Convert kilolitres to million litres (1 million litres = 1,000 kilolitres)
canada_clean["canada_total_ml"] = canada_clean["VALUE"] / 1000

canada_clean.head(3)

,REF_DATE,VALUE,date,canada_total_ml
723,1976-01,503352.0,1976-01-01,503.352
797,1976-02,461721.0,1976-02-01,461.721
871,1976-03,555739.0,1976-03-01,555.739


In [74]:
# One row per month, with the date as the row label
canada_wide = canada_clean.set_index("date")[["canada_total_ml"]]

# Missing values and summary statistics
print(canada_wide.isna().sum(), "\n")
canada_wide.describe()

canada_total_ml    0
dtype: int64 



,canada_total_ml
count,606.000000
mean,650.518906
std,82.447795
min,418.385000
25%,598.385500
50%,639.335500
75%,688.015750
max,853.532000


#### Cleaning summary

I kept only total milk sold off farms, converted `REF_DATE` to dates and converted kilolitres to million litres. The result, `canada_wide`, has 606 rows (January 1976 to June 2026) and one column, `canada_total_ml`, with no missing values.